# Paper 2 — Cluster-Aware Medical Report Summarization with Hybrid RAG

**Research implementation and reproduction notebook**  
*Cluster-Aware Retrieval-Augmented Generation (RAG) with Hybrid Retrieval for Faithful Medical Report Summarization* — 39th Canadian Conference on Artificial Intelligence (PMLR 318, 2026).

**Authors:** Kiarash Torabizadeh, Rachid Hedjam, Mebarka Allaoui, and Bessam Abdulrazak.  
**Affiliations (as published):** Bishop's University, Sherbrooke, Canada (Torabizadeh, Hedjam, Allaoui); Université de Sherbrooke, Canada (Abdulrazak); Université du Québec en Outaouais (UQO) is additionally listed for Hedjam.  
**Publication:** [Paper 2 (PMLR)](https://proceedings.mlr.press/v318/torabizadeh26a.html) · [Related Paper 1 (IEEE DOI)](https://doi.org/10.1109/ACDSA65407.2025.11165844)

## Purpose and workflow

This notebook implements **evidence-grounded, cluster-level summarization** of a *synthetic hypertension dataset*. It (1) builds a document corpus from cluster profiles and patient report snippets; (2) indexes text using TF-IDF and sentence embeddings; (3) retrieves cluster-relevant evidence with a weighted hybrid score; (4) sweeps candidate retrieval weights; (5) generates evidence-cited summaries; and (6) evaluates summaries against the S1 (true-label) and S4 (t-SNE-PSO) non-RAG baselines from Paper 1.

> **Research use only.** The dataset is synthetic; the generated summaries are not medical advice. The automated claim-support measure relies on **lexical TF-IDF similarity**, not clinical entailment. Results may change across API runs even when the retrieval evidence is unchanged.

### Running this notebook

1. Open in **Google Colab** and use **Runtime → Run all** in order, after providing the required files below.
2. Upload the input files to the Colab `/content/` working directory, using the **Files** sidebar. Keep the original filenames.
3. Add an `OPENAI_API_KEY` secret under **Colab → Secrets**, and enable notebook access for that secret. API calls may incur charges. Never paste a real API key into the notebook.
4. Allow the sentence-transformers installation cell to complete; then run retrieval, the alpha sweep, generation, and evaluation cells sequentially.

**Required files** (in `/content/`):

| Input | Role |
|---|---|
| `cluster_profile_docs.md` | Three structured cluster-profile documents |
| `snippet_bank_reports.jsonl` | Patient-report snippet bank |
| `cluster_profile_table.csv` | Numerical cluster features for completeness scoring |
| `step7_eval_input_template.json` | Three-row evaluation JSON template |
| `S1_TRUE_L_patient_dataset_with_reports_2_x.xlsx` | Paper 1 S1 cluster-summary text boxes |
| `S4_TSNE-PSO-patient_dataset_with_report_clustering_tsn-pso.xlsx` | Paper 1 S4 cluster-summary text boxes |

**Execution order:** evidence corpus → TF-IDF/embedding indexes → retriever → evidence prompt → API credentials → alpha sweep → final summaries → S1/S4 references → Step 7 evaluations → baselines → deltas.

---

## 1. Construct the evidence corpus

Load the cluster-profile documents and patient report snippets into a common list named `docs`. Each document has `doc_id`, `risk_group`, `doc_type` (`profile` or `snippet`), and `text` fields. This corpus is the input to both retrieval indexes.

**Data count:** do not assume that the comment in an older notebook ("150 snippets") matches a released input file. The provided snippet bank examined during this project contained **149 snippets plus 3 profiles (152 documents)**; the following cell prints the actual loaded count.

In [ ]:
import json, re
import pandas as pd

PROFILE_MD = "cluster_profile_docs.md"
SNIPPETS_JSONL = "snippet_bank_reports.jsonl"

def load_profile_docs(md_path):
    # Extract text blocks inside ```text ... ```
    with open(md_path, "r", encoding="utf-8") as f:
        md = f.read()

    blocks = re.findall(r"```text\n(.*?)```", md, flags=re.DOTALL)
    docs = []
    for b in blocks:
        b = b.strip()
        # Expect first line like: "CLUSTER PROFILE — low_risk (n=...)"
        m = re.search(r"CLUSTER PROFILE —\s*(\w+)", b)
        risk = m.group(1) if m else "unknown"
        docs.append({
            "doc_id": f"profile_{risk}",
            "risk_group": risk,
            "doc_type": "profile",
            "text": re.sub(r"\s+", " ", b).strip()
        })
    return docs

def load_snippet_docs(jsonl_path):
    docs = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            docs.append({
                "doc_id": row["snippet_id"],
                "risk_group": row["risk_group"],
                "doc_type": "snippet",
                "text": re.sub(r"\s+", " ", row["snippet_text"]).strip()
            })
    return docs

docs = load_profile_docs(PROFILE_MD) + load_snippet_docs(SNIPPETS_JSONL)
print("Total docs:", len(docs))  # should be 153


## 2. Build the TF-IDF retrieval index

Fit a **word-unigram and bigram** TF-IDF vectorizer over the full evidence corpus. At query time, the resulting matrix is used to measure lexical cosine similarity between a query and candidate documents. This captures shared terms such as *blood pressure*, *glucose*, and *cholesterol*.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

texts = [d["text"] for d in docs]

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1,2),     # helps capture phrases
    min_df=1
)
X_tfidf = tfidf.fit_transform(texts)  # sparse (N_docs × vocab)


## 3. Build the sentence-embedding retrieval index

Install `sentence-transformers`, then encode every evidence document with `all-MiniLM-L6-v2`. The embeddings are normalized so their dot product can be interpreted as cosine similarity.

> Installation is performed in the next cell using a Colab/Jupyter `%pip` command. This cell requires an internet connection on the first model download.

In [ ]:
%pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")  # small, fast, common
E = model.encode(texts, normalize_embeddings=True)  # (N_docs × dim)


## 4. Normalize retrieval scores and combine lexical and semantic similarity

TF-IDF and embedding similarities may have different value ranges. The implementation **min–max normalizes each score vector within the candidate set for the current query and risk group**, then computes:

\[
\widehat{s}_i = \frac{s_i - \min(s)}{\max(s)-\min(s)+\varepsilon}
\]

\[
s_{\mathrm{hybrid}}(q,d)=\alpha\,\widehat{s}_{\mathrm{TF-IDF}}(q,d)+(1-\alpha)\,\widehat{s}_{\mathrm{embedding}}(q,d).
\]

- **α = 0:** embedding-only ranking.
- **α = 1:** TF-IDF-only ranking.
- **0 < α < 1:** weighted hybrid ranking.

The `hybrid_retrieve` function restricts candidates to the chosen `risk_group` before ranking and returns up to `top_k` documents, together with component and combined scores.

In [ ]:
import numpy as np

def minmax_norm(a, eps=1e-9):
    a = np.asarray(a).astype(float)
    return (a - a.min()) / (a.max() - a.min() + eps)

def hybrid_retrieve(query, risk_group=None, top_k=8, alpha=0.25):
    # 1) Candidate filtering (optional but recommended)
    idxs = list(range(len(docs)))
    if risk_group is not None:
        idxs = [i for i in idxs if docs[i]["risk_group"] == risk_group]

    # 2) TF-IDF similarity
    q_t = tfidf.transform([query])
    s_tfidf = cosine_similarity(q_t, X_tfidf[idxs]).ravel()

    # 3) Embedding similarity
    q_e = model.encode([query], normalize_embeddings=True)
    s_emb = (E[idxs] @ q_e[0])  # cosine since normalized

    # 4) Normalize and combine
    s_t = minmax_norm(s_tfidf)
    s_e = minmax_norm(s_emb)
    s_h = alpha * s_t + (1 - alpha) * s_e

    # 5) Rank
    order = np.argsort(-s_h)[:top_k]
    results = []
    for j in order:
        i = idxs[j]
        results.append({
            "doc_id": docs[i]["doc_id"],
            "risk_group": docs[i]["risk_group"],
            "doc_type": docs[i]["doc_type"],
            "hybrid": float(s_h[j]),
            "tfidf": float(s_t[j]),
            "emb": float(s_e[j]),
            "text": docs[i]["text"]
        })
    return results


## 5. Define the clinical retrieval query

The query combines hypertension-related concepts and clinical indicators used in the study. It is applied consistently during retrieval for each risk group so that changes in `alpha` reflect changes in retrieval weighting rather than changes in the query.

In [ ]:
query = ("hypertension blood pressure systolic diastolic BMI obesity "
         "cholesterol glucose diabetes risk factors cardiovascular")

## 6. Assemble citation-labeled evidence and the generation prompt

For each risk group, `build_evidence_block` supplies a **cluster profile** as `[C]` and up to `top_k` ranked retrieved documents as `[S1]`, `[S2]`, etc. `build_user_prompt` then requests a structured summary covering the cluster overview, clinical indicators, risk factors, and non-diagnostic follow-ups.

**Implementation note:** the current retriever ranks both profile and snippet documents. Because the profile is *also* inserted explicitly as `[C]`, it may appear a second time under an `[S#]` label. This reflects the original implementation and is **not silently changed in this documentation-only revision**. For a strict "profile + top-k snippets" design, candidate filtering should exclude `doc_type == "profile"` before selecting the snippets.

### Retrieval parameters and interpretation

The evaluation tests `alpha ∈ {0, 0.25, 0.50, 0.75, 1.00}` with **`top_k = 8`**. Values closer to zero emphasize semantic-embedding similarity; values closer to one emphasize matching clinical terminology through TF-IDF. The optimal setting is **measured by the evaluation procedure**, rather than assumed in advance.

**Caution:** the published Paper 2 reported **α = 0.25**. Later runs may select a different value. In this notebook, `best_alpha` is computed from the current sweep and passed explicitly to the final retrieval calls.

### Evidence block and user-prompt construction

The following functions create the evidence text and prompt used by the model. They are reused both in the alpha sweep and in the subsequent cluster-specific summary-generation cells.

In [ ]:
def build_evidence_block(cluster_name, top_k=8, alpha=0.25):
    query = ("hypertension blood pressure systolic diastolic BMI obesity "
             "cholesterol glucose diabetes risk factors cardiovascular")

    # 1) Rank up to top-k evidence documents within the specified risk group.
    snips = hybrid_retrieve(query, risk_group=cluster_name, top_k=top_k, alpha=alpha)

    # 2) Retrieve the structured profile for the same risk group.
    prof = next(d for d in docs if d["doc_type"]=="profile" and d["risk_group"]==cluster_name)

    # 3) Assemble citation-labeled evidence for the language model.
    lines = []
    lines.append("EVIDENCE (You MUST cite these IDs; do not invent new ones)\n")
    lines.append(f"[C] Cluster profile ({cluster_name}):\n{prof['text']}\n")

    for i, r in enumerate(snips, start=1):
        lines.append(f"[S{i}] Patient snippet:\n{r['text']}\n")

    return "\n".join(lines)

def build_user_prompt(cluster_name, evidence_block):
    return f"""Task: Write a cluster-level summary for cluster: {cluster_name}.

Output format (exact headings):
Cluster Overview
- (2–4 bullet sentences with citations)

Key Indicators (BP / BMI / Cholesterol / Glucose / Kidney)
- Blood Pressure: ...
- BMI: ...
- Cholesterol: ...
- Glucose: ...
- Kidney: ... (if not present → "insufficient evidence")

Main Risk Factors
- (3–6 bullet sentences with citations)

Suggested follow-ups (non-diagnostic, generic, grounded)
- (3–5 bullets). Each bullet must be justified by observed indicators (cite [C] and/or snippets).
- No medication advice. No diagnoses.

EVIDENCE:
{evidence_block}
"""


### Configure OpenAI credentials securely

Use a Colab secret named **`OPENAI_API_KEY`** (Secrets panel → **Add new secret** → enable notebook access). The next cell retrieves that secret without printing or storing the key in the notebook. Outside Colab, set the `OPENAI_API_KEY` environment variable before launching Jupyter.

**Never paste an API key into notebook source, saved outputs, screenshots, or Git history.**

In [ ]:
import os

# Colab: obtain the key from its Secrets store rather than embedding it in source.
if not os.getenv("OPENAI_API_KEY"):
    try:
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
    except ImportError:
        pass  # Non-Colab environments should set the environment variable.
    except Exception as exc:
        raise RuntimeError(
            "Unable to read the OPENAI_API_KEY Colab secret. "
            "Add the secret and enable notebook access."
        ) from exc

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY is not configured. Use Colab Secrets or an environment variable."
    )


## 7. Sweep candidate alpha values and select a retrieval weight

For **each alpha and each of the three risk groups**, the notebook retrieves evidence, generates a GPT summary, and calculates automatic metrics. It writes:

- `alpha_sweep_results.csv` — one row per **alpha × cluster**;
- `alpha_sweep_macro.csv` — macro averages over the three groups for each alpha;
- `alpha_sweep_outputs.json` — complete generated summaries and evidence blocks for auditing.

**Selection priority:** highest macro supported-claim rate, then highest completeness, then fewest absolute-claim flags. The first row of `macro_sorted` is stored as `best_alpha`.

**Reproducibility:** `temperature=0` reduces sampling variability, but does not guarantee identical API generations across runs. The support checker is based on TF-IDF similarity at a **0.22 threshold**, which should not be interpreted as verified factual entailment.

In [ ]:
# =========================
# Alpha sweep: evaluate candidate lexical/semantic retrieval weights.
# alpha=0.0  -> embedding-only retrieval
# alpha=1.0  -> TF-IDF-only retrieval
# (because hybrid score uses: s_h = alpha*s_tfidf + (1-alpha)*s_emb)
# =========================

import time
import pandas as pd
import numpy as np
import re, json, os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

# ---- Experimental settings (held fixed across alpha values) ----
ALPHAS = [0.0, 0.25, 0.5, 0.75, 1.0]
CLUSTERS = ["low_risk", "average_risk", "high_risk"]
TOP_K = 8
MODEL_NAME = "gpt-4o-mini"
TEMPERATURE = 0  # important: reduce randomness when comparing alphas

client = OpenAI()
assert os.getenv("OPENAI_API_KEY"), "Please set OPENAI_API_KEY before running this cell."

# ---- Step 7-style automatic evaluator (no reference text required) ----
PROFILE_TABLE = "cluster_profile_table.csv"
profiles = pd.read_csv(PROFILE_TABLE)

BP_FAMILY = ["pct_BP_elevated_or_higher", "pct_BP_stage2"]
MET_FAMILY = ["pct_BMI_overweight", "pct_BMI_obese",
              "pct_Glucose_prediabetes", "pct_Glucose_diabetes",
              "pct_Chol_borderline", "pct_Chol_high"]

INDICATOR_KEYWORDS = {
    "pct_BP_elevated_or_higher": ["blood pressure", "bp", "hypertension", "systolic", "diastolic", "elevated bp"],
    "pct_BP_stage2": ["stage 2", "severe hypertension", "very high blood pressure", "140/90"],
    "pct_BMI_overweight": ["overweight", "bmi", "weight"],
    "pct_BMI_obese": ["obesity", "obese", "bmi"],
    "pct_Glucose_prediabetes": ["glucose", "prediabet", "blood sugar", "elevated sugar"],
    "pct_Glucose_diabetes": ["diabetes", "diabetic", "hyperglyc", "very high glucose"],
    "pct_Chol_borderline": ["cholesterol", "borderline cholesterol", "lipid"],
    "pct_Chol_high": ["high cholesterol", "hypercholesterol", "cholesterol", "lipid"]
}

ABSOLUTE_PATTERNS = [r"\ball patients\b", r"\bevery patient\b", r"\ball individuals\b", r"\bnone of the patients\b",
                     r"\balways\b", r"\bnever\b"]
DIAGNOSIS_PATTERNS = [r"\bdiagnos(ed|is)\b", r"\bchronic kidney disease\b", r"\bckd\b"]
MEDICATION_PATTERNS = [r"\bprescribe\b", r"\bmedication\b", r"\bdosage\b", r"\bstatin\b", r"\binsulin\b", r"\bmetformin\b"]

def parse_evidence_block(evidence_block: str):
    pattern = r"(\[(?:C|S\d+)\])"
    parts = re.split(pattern, evidence_block)
    items = []
    cur_id = None
    cur_text = []
    for p in parts:
        if not p:
            continue
        if re.fullmatch(pattern, p):
            if cur_id is not None:
                txt = re.sub(r"\s+", " ", "\n".join(cur_text)).strip()
                if txt:
                    items.append((cur_id, txt))
            cur_id = p
            cur_text = []
        else:
            cur_text.append(p.strip())
    if cur_id is not None:
        txt = re.sub(r"\s+", " ", "\n".join(cur_text)).strip()
        if txt:
            items.append((cur_id, txt))
    cleaned = []
    for eid, txt in items:
        txt = re.sub(r"^Cluster profile.*?:\s*", "", txt, flags=re.IGNORECASE)
        txt = re.sub(r"^Patient snippet.*?:\s*", "", txt, flags=re.IGNORECASE)
        cleaned.append((eid, txt.strip()))
    return cleaned

def extract_claims(summary_text: str, min_chars=20, max_claims=12):
    lines = [ln.strip() for ln in summary_text.splitlines() if ln.strip()]
    bullet_lines = [ln for ln in lines if ln.startswith(("-", "•", "*"))]
    text_no_cite = re.sub(r"\[(?:C|S\d+)\]", "", summary_text)
    text_no_cite = re.sub(r"\s+", " ", text_no_cite).strip()
    claims = []
    if bullet_lines:
        for ln in bullet_lines:
            ln = re.sub(r"^[-•*]\s*", "", ln)
            ln = re.sub(r"\[(?:C|S\d+)\]", "", ln)
            ln = re.sub(r"\s+", " ", ln).strip()
            if len(ln) >= min_chars:
                claims.append(ln)
    else:
        sents = re.split(r"(?<=[\.\!\?])\s+", text_no_cite)
        claims = [s.strip() for s in sents if len(s.strip()) >= min_chars]
    return claims[:max_claims]

def support_check_claims(claims, evidence_items, sim_threshold=0.22):
    e_ids = [eid for eid,_ in evidence_items]
    e_texts = [txt for _,txt in evidence_items]
    v = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
    X = v.fit_transform(e_texts + claims)
    E_ = X[:len(e_texts)]
    C_ = X[len(e_texts):]
    sims = cosine_similarity(C_, E_)  # (n_claims x n_evidence)
    results = []
    for i, claim in enumerate(claims):
        j = int(np.argmax(sims[i]))
        best_sim = float(sims[i, j])
        results.append({
            "supported": best_sim >= sim_threshold,
            "best_evidence_id": e_ids[j],
            "best_similarity": best_sim
        })
    return results

def pick_must_mentions(cluster_name):
    row = profiles.loc[profiles["risk_group"]==cluster_name].iloc[0]
    bp_best = max(BP_FAMILY, key=lambda c: row[c])
    met_best = max(MET_FAMILY, key=lambda c: row[c])
    return [bp_best, met_best]

def completeness_score(summary_text, must_inds):
    s = summary_text.lower()
    hits = 0
    for ind in must_inds:
        kws = INDICATOR_KEYWORDS.get(ind, [])
        if any(kw in s for kw in kws):
            hits += 1
    return hits / max(1, len(must_inds))

def count_patterns(text, patterns):
    t = text.lower()
    return sum(len(re.findall(pat, t)) for pat in patterns)

def evaluate_rag_summary(cluster_name, rag_summary, evidence_block):
    evidence_items = parse_evidence_block(evidence_block)
    claims = extract_claims(rag_summary, max_claims=12)
    checks = support_check_claims(claims, evidence_items, sim_threshold=0.22)
    supported = sum(1 for r in checks if r["supported"])
    total = len(checks) if len(checks)>0 else 1
    supported_rate = 100.0 * supported / total
    unsupported_rate = 100.0 - supported_rate

    must_inds = pick_must_mentions(cluster_name)
    comp = 100.0 * completeness_score(rag_summary, must_inds)

    abs_ct = count_patterns(rag_summary, ABSOLUTE_PATTERNS)
    diag_ct = count_patterns(rag_summary, DIAGNOSIS_PATTERNS)
    med_ct = count_patterns(rag_summary, MEDICATION_PATTERNS)

    return {
        "cluster": cluster_name,
        "supported_claim_rate_%": supported_rate,
        "unsupported_claim_rate_%": unsupported_rate,
        "completeness_%": comp,
        "absolute_claim_flags": abs_ct,
        "diagnosis_like_flags": diag_ct,
        "medication_flags": med_ct,
    }

# ---- Generate a citation-constrained summary for one cluster ----
def generate_one_cluster(cluster_name, alpha):
    evidence_block = build_evidence_block(cluster_name, top_k=TOP_K, alpha=alpha)
    system_text = """You are a medical summarization assistant for research. Use ONLY the provided EVIDENCE.
    Hard rules:
    1) Every major claim MUST have at least one citation like [C] or [S3]. Place citations at the end of the sentence.
    2) Do NOT diagnose diseases. Do NOT prescribe medication. Do NOT claim causality.
    3) Use hedged language: "many", "often", "tends to", "on average", "a subset".
    4) If evidence is missing or unclear, write: "insufficient evidence" and cite [C] if relevant.
    5) Do not invent numbers. If you mention a number, it must appear in the evidence and be cited.
    Only output in the requested format.
    """
    user_text = build_user_prompt(cluster_name, evidence_block)
    resp = client.responses.create(
        model=MODEL_NAME,
        temperature=TEMPERATURE,
        input=[
            {"role":"system", "content": system_text},
            {"role":"user", "content": user_text},
        ],
    )
    return resp.output_text, evidence_block

# ---- Evaluate each alpha across all three risk groups ----
rows = []
outputs = []  # retain full generated summaries and evidence for auditability
for a in ALPHAS:
    for c in CLUSTERS:
        summary_text, ev_block = generate_one_cluster(c, a)
        m = evaluate_rag_summary(c, summary_text, ev_block)
        m["alpha"] = a
        rows.append(m)
        outputs.append({
            "alpha": a,
            "cluster": c,
            "summary": summary_text,
            "evidence_block": ev_block
        })
        time.sleep(0.2)  # light pacing (optional)

df_alpha = pd.DataFrame(rows)
df_alpha.to_csv("alpha_sweep_results.csv", index=False)

# Compute macro-average metrics across the three risk groups.
macro = df_alpha.groupby("alpha")[[
    "supported_claim_rate_%","unsupported_claim_rate_%","completeness_%",
    "absolute_claim_flags","diagnosis_like_flags","medication_flags"
]].mean().reset_index()

macro.to_csv("alpha_sweep_macro.csv", index=False)

# Rank by claim support, completeness, then fewer absolute-claim flags.
macro_sorted = macro.sort_values(
    by=["supported_claim_rate_%","completeness_%","absolute_claim_flags"],
    ascending=[False, False, True]
)

print("Saved: alpha_sweep_results.csv and alpha_sweep_macro.csv")
display(macro_sorted)

# Preserve exact generated summaries and evidence for qualitative review.
with open("alpha_sweep_outputs.json", "w", encoding="utf-8") as f:
    json.dump(outputs, f, indent=2)
print("Saved: alpha_sweep_outputs.json")


# ============================================================
# Report the highest-ranked alpha and its retrieval-weight interpretation.
# ============================================================

best = macro_sorted.iloc[0]
best_alpha = float(best["alpha"])

print("\n" + "=" * 55)
print(f"BEST ALPHA: {best_alpha:.2f}")
print("=" * 55)

print(f"Supported-claim rate : {best['supported_claim_rate_%']:.2f}%")
print(f"Unsupported claims   : {best['unsupported_claim_rate_%']:.2f}%")
print(f"Completeness         : {best['completeness_%']:.2f}%")
print(f"Absolute-claim flags : {best['absolute_claim_flags']:.2f}")
print(f"Diagnosis-like flags : {best['diagnosis_like_flags']:.2f}")
print(f"Medication flags     : {best['medication_flags']:.2f}")

print("\nRetrieval interpretation:")
print(f"TF-IDF weight        : {best_alpha * 100:.0f}%")
print(f"Embedding weight     : {(1 - best_alpha) * 100:.0f}%")
print("=" * 55)



### Selected alpha and run-level interpretation

The sweep displays the ranking and prints the chosen `best_alpha` and its corresponding TF-IDF/embedding weights. The selected value is a **run-specific experimental outcome**, not a permanent or clinically validated optimum.

## 8. Generate the final risk-group summaries

The next three code sections generate new summaries for **high**, **low**, and **average** risk groups using `alpha=best_alpha`, the same `MODEL_NAME`, and `TEMPERATURE` as the sweep.

**Important:** these are **new API calls**, not the exact texts already evaluated during the alpha sweep. Their wording and scores may differ. For strict consistency between the alpha-selection score and the final Step 7 evaluation, reuse the winning entries from `alpha_sweep_outputs.json` instead of regenerating them.

### 8.1 High-risk summary

Generate the high-risk report from the current cluster profile and ranked evidence; save the response in `high_resp`. The following cell stores its exact evidence block as `Evidence_block_high`.

In [ ]:
import os
from openai import OpenAI

client = OpenAI()

# OPENAI_API_KEY is loaded securely in the credentials cell above.
assert os.getenv('OPENAI_API_KEY'), 'Please set OPENAI_API_KEY before running this cell.'

system_text = """You are a medical summarization assistant for research. Use ONLY the provided EVIDENCE.
Hard rules:
1) Every major claim MUST have at least one citation like [C] or [S3]. Place citations at the end of the sentence.
2) Do NOT diagnose diseases. Do NOT prescribe medication. Do NOT claim causality.
3) Use hedged language: "many", "often", "tends to", "on average", "a subset".
4) If evidence is missing or unclear, write: "insufficient evidence" and cite [C] if relevant.
5) Do not invent numbers. If you mention a number, it must appear in the evidence and be cited.
Only output in the requested format.
"""
evidence = build_evidence_block("high_risk", top_k=8, alpha=best_alpha)
user_text = build_user_prompt("high_risk", evidence)

high_resp = client.responses.create(
    model="gpt-4o-mini",
    temperature=TEMPERATURE,
    input=[
        {"role":"system", "content": system_text},
        {"role":"user", "content": user_text},
    ],
)

print(high_resp.output_text)


In [ ]:
Evidence_block_high = evidence
print(Evidence_block_high)

### 8.2 Low-risk summary

Generate the low-risk report using the selected retrieval weight; save the response in `low_resp` and the associated evidence block as `Evidence_block_low`.

In [ ]:
import os
from openai import OpenAI

client = OpenAI()

# OPENAI_API_KEY is loaded securely in the credentials cell above.
assert os.getenv('OPENAI_API_KEY'), 'Please set OPENAI_API_KEY before running this cell.'

system_text = """You are a medical summarization assistant for research. Use ONLY the provided EVIDENCE.
Hard rules:
1) Every major claim MUST have at least one citation like [C] or [S3]. Place citations at the end of the sentence.
2) Do NOT diagnose diseases. Do NOT prescribe medication. Do NOT claim causality.
3) Use hedged language: "many", "often", "tends to", "on average", "a subset".
4) If evidence is missing or unclear, write: "insufficient evidence" and cite [C] if relevant.
5) Do not invent numbers. If you mention a number, it must appear in the evidence and be cited.
Only output in the requested format.
"""
evidence = build_evidence_block("low_risk", top_k=8, alpha=best_alpha)
user_text = build_user_prompt("low_risk", evidence)

low_resp = client.responses.create(
    model="gpt-4o-mini",
    temperature=TEMPERATURE,
    input=[
        {"role":"system", "content": system_text},
        {"role":"user", "content": user_text},
    ],
)

print(low_resp.output_text)


In [ ]:
Evidence_block_low = evidence
print(Evidence_block_low)

### 8.3 Average-risk summary

Generate the average-risk report using the selected retrieval weight; save the response in `average_resp` and the associated evidence block as `Evidence_block_average`.

In [ ]:
import os
from openai import OpenAI

client = OpenAI()

# OPENAI_API_KEY is loaded securely in the credentials cell above.
assert os.getenv('OPENAI_API_KEY'), 'Please set OPENAI_API_KEY before running this cell.'

system_text = """You are a medical summarization assistant for research. Use ONLY the provided EVIDENCE.
Hard rules:
1) Every major claim MUST have at least one citation like [C] or [S3]. Place citations at the end of the sentence.
2) Do NOT diagnose diseases. Do NOT prescribe medication. Do NOT claim causality.
3) Use hedged language: "many", "often", "tends to", "on average", "a subset".
4) If evidence is missing or unclear, write: "insufficient evidence" and cite [C] if relevant.
5) Do not invent numbers. If you mention a number, it must appear in the evidence and be cited.
Only output in the requested format.
"""
evidence = build_evidence_block("average_risk", top_k=8, alpha=best_alpha)
user_text = build_user_prompt("average_risk", evidence)

average_resp = client.responses.create(
    model="gpt-4o-mini",
    temperature=TEMPERATURE,
    input=[
        {"role":"system", "content": system_text},
        {"role":"user", "content": user_text},
    ],
)

print(average_resp.output_text)


In [ ]:
Evidence_block_average = evidence
print(Evidence_block_average)

## 9. Construct the S1 and S4 comparison inputs

The next cell loads `step7_eval_input_template.json` and creates two evaluation inputs in `/content/`:

| Output file | `rag_summary` | `reference_s1` | `evidence_block` |
|---|---|---|---|
| `step7_eval_input_S1vsRAG.json` | Current RAG summary | **S1 overall cluster summary** from its Excel text box | Current retrieved evidence |
| `step7_eval_input_S4vsRAG.json` | **The same** current RAG summary | **S4 overall cluster summary** from its Excel text box | **The same** current retrieved evidence |

The code reads the **overall summary text boxes** stored in the Excel drawings, **not** the individual `Clinical_Report` rows. Numeric cluster labels are mapped to low/average/high risks independently for S1 and S4.

**Interpretation:** these files evaluate one *shared* RAG generation against two different Paper 1 reference summaries. They do **not** establish separately generated S1-RAG and S4-RAG retrieval pipelines. The field name `reference_s1` is retained for evaluator compatibility even when it contains the S4 reference.

In [ ]:

import json
import re
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path
from copy import deepcopy

# ============================================================
# Create S1-vs-RAG and S4-vs-RAG evaluation inputs
# using the overall cluster summaries from Excel text boxes
# ============================================================

DATA_DIR = Path("/content")

TEMPLATE_FILE = DATA_DIR / "step7_eval_input_template.json"

S1_FILE = DATA_DIR / "S1_TRUE_L_patient_dataset_with_reports_2_x.xlsx"
S4_FILE = DATA_DIR / "S4_TSNE-PSO-patient_dataset_with_report_clustering_tsn-pso.xlsx"

S1_OUTPUT = DATA_DIR / "step7_eval_input_S1vsRAG.json"
S4_OUTPUT = DATA_DIR / "step7_eval_input_S4vsRAG.json"

# 1. Check input files
for path in [TEMPLATE_FILE, S1_FILE, S4_FILE]:
    if not path.is_file():
        raise FileNotFoundError(f"Missing input file: {path}")

# 2. Load template
with open(TEMPLATE_FILE, "r", encoding="utf-8") as f:
    template = json.load(f)

expected_clusters = {"low_risk", "average_risk", "high_risk"}

if (
    len(template) != 3
    or {item["cluster"] for item in template} != expected_clusters
):
    raise ValueError("Template must contain exactly the three risk groups.")

# 3. Get the latest generated RAG summaries and evidence
rag_data = {
    "low_risk": {
        "rag_summary": low_resp.output_text,
        "evidence_block": Evidence_block_low
    },
    "average_risk": {
        "rag_summary": average_resp.output_text,
        "evidence_block": Evidence_block_average
    },
    "high_risk": {
        "rag_summary": high_resp.output_text,
        "evidence_block": Evidence_block_high
    }
}

# Check that the generated values exist
for cluster, fields in rag_data.items():
    for field_name, value in fields.items():
        if not isinstance(value, str) or not value.strip():
            raise ValueError(
                f"Missing or empty {field_name} for {cluster}"
            )

# 4. Extract the original cluster-level summaries from Excel text boxes.
# Excel stores drawing text boxes in its internal XML files.
# These are overall summaries, not concatenated patient-report rows.

XDR = "{http://schemas.openxmlformats.org/drawingml/2006/spreadsheetDrawing}"
A = "{http://schemas.openxmlformats.org/drawingml/2006/main}"

def extract_cluster_summaries(xlsx_path, label_map):

    summaries_by_label = {}

    with zipfile.ZipFile(xlsx_path, "r") as archive:

        drawing_files = [
            name for name in archive.namelist()
            if re.fullmatch(
                r"xl/drawings/drawing\d+\.xml",
                name
            )
        ]

        for drawing_file in drawing_files:

            root = ET.fromstring(archive.read(drawing_file))

            for shape in root.findall(f".//{XDR}sp"):

                paragraphs = []

                for paragraph in shape.findall(f".//{A}p"):

                    text = "".join(
                        t.text or ""
                        for t in paragraph.findall(f".//{A}t")
                    )

                    if text.strip():
                        paragraphs.append(text.strip())

                full_text = "\n".join(paragraphs).strip()

                # Find labels such as "Cluster 0 (low):"
                match = re.match(
                    r"^Cluster\s+(\d+)\b",
                    full_text,
                    flags=re.IGNORECASE
                )

                if not match:
                    continue

                label = int(match.group(1))

                if ":" not in full_text:
                    raise ValueError(
                        f"Missing heading separator for Cluster {label}"
                    )

                # Keep only the summary after the colon
                summary = full_text.split(":", 1)[1].strip()

                if not summary:
                    raise ValueError(
                        f"Empty overall summary for Cluster {label}"
                    )

                if label in summaries_by_label:
                    raise ValueError(
                        f"Duplicate summary for Cluster {label}"
                    )

                summaries_by_label[label] = summary

    if set(summaries_by_label) != set(label_map):
        raise ValueError(
            f"Unexpected cluster labels in {xlsx_path.name}. "
            f"Found: {list(summaries_by_label)}"
        )

    # Translate numeric cluster labels to risk group names
    return {
        risk_name: summaries_by_label[label]
        for label, risk_name in label_map.items()
    }

# 5. Extract S1 overall cluster summaries
s1_references = extract_cluster_summaries(
    S1_FILE,
    {
        0: "low_risk",
        1: "average_risk",
        2: "high_risk"
    }
)

# 6. Extract S4 t-SNE-PSO overall cluster summaries
s4_references = extract_cluster_summaries(
    S4_FILE,
    {
        0: "average_risk",
        1: "low_risk",
        2: "high_risk"
    }
)

# 7. Create each Step 7 JSON input file
def create_eval_input(references, output_file):

    items = deepcopy(template)

    for item in items:

        cluster = item["cluster"]

        # Insert current Hybrid RAG summary
        item["rag_summary"] = rag_data[cluster]["rag_summary"]

        # Insert current Hybrid RAG evidence
        item["evidence_block"] = rag_data[cluster]["evidence_block"]

        # Insert original overall S1 or S4 cluster summary
        # The field name is retained for evaluator compatibility.
        item["reference_s1"] = references[cluster]

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(items, f, indent=2, ensure_ascii=False)

    print(f"Saved: {output_file.name}")

    return items

s1_items = create_eval_input(
    s1_references,
    S1_OUTPUT
)

s4_items = create_eval_input(
    s4_references,
    S4_OUTPUT
)

# 8. Print short previews to verify the selected reference summaries.
for cluster in ["low_risk", "average_risk", "high_risk"]:
    print(f"\n--- {cluster} ---")
    print("S1 reference:", s1_references[cluster][:110], "...")
    print("S4 reference:", s4_references[cluster][:110], "...")

print("\nBoth files are saved in /content/")
print("They are ready for the Step 7 evaluation cells.")


## 10. Evaluate the RAG summaries with the S1 reference

Read `step7_eval_input_S1vsRAG.json` and calculate: (i) TF-IDF similarity between the generated RAG summary and the original S1 cluster summary; (ii) lexical claim-support rate against retrieved evidence; (iii) required-indicator completeness; and (iv) rule-based safety/overreach flags.

**Output:** `step7_evaluation_results_S1vsRAG.csv`. Only the *reference similarity* is specific to S1 in this shared-RAG implementation.

In [ ]:
#!/usr/bin/env python3
import json, re
import pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROFILE_TABLE = "cluster_profile_table.csv"

profiles = pd.read_csv(PROFILE_TABLE)

BP_FAMILY = ["pct_BP_elevated_or_higher", "pct_BP_stage2"]
MET_FAMILY = ["pct_BMI_overweight", "pct_BMI_obese", "pct_Glucose_prediabetes", "pct_Glucose_diabetes", "pct_Chol_borderline", "pct_Chol_high"]

INDICATOR_KEYWORDS = {
    "pct_BP_elevated_or_higher": ["blood pressure", "bp", "hypertension", "pre-hypertens", "systolic", "diastolic", "elevated bp"],
    "pct_BP_stage2": ["stage 2", "severe hypertension", "very high blood pressure", "140/90"],
    "pct_BMI_overweight": ["overweight", "bmi", "weight"],
    "pct_BMI_obese": ["obesity", "obese", "bmi"],
    "pct_Glucose_prediabetes": ["glucose", "prediabet", "blood sugar", "elevated sugar"],
    "pct_Glucose_diabetes": ["diabetes", "diabetic", "hyperglyc", "very high glucose"],
    "pct_Chol_borderline": ["cholesterol", "borderline cholesterol", "lipid"],
    "pct_Chol_high": ["high cholesterol", "hypercholesterol", "cholesterol", "lipid"]
}

ABSOLUTE_PATTERNS = [r"\ball patients\b", r"\bevery patient\b", r"\ball individuals\b", r"\bnone of the patients\b", r"\balways\b", r"\bnever\b"]
DIAGNOSIS_PATTERNS = [r"\bdiagnos(ed|is)\b", r"\bhas (?:a|an|the)\b.*\b(disease|disorder|condition)\b", r"\bchronic kidney disease\b", r"\bckd\b", r"\bmyocardial infarction\b", r"\bstroke\b"]
MEDICATION_PATTERNS = [r"\bprescribe\b", r"\bmedication\b", r"\bdrug\b", r"\bdosage\b", r"\bmetformin\b", r"\bstatin\b", r"\binsulin\b", r"\bantihypertensive\b"]

def parse_evidence_block(evidence_block: str):
    pattern = r"(\[(?:C|S\d+)\])"
    parts = re.split(pattern, evidence_block)
    items = []
    cur_id = None
    cur_text = []
    for p in parts:
        if not p:
            continue
        if re.fullmatch(pattern, p):
            if cur_id is not None:
                txt = re.sub(r"\s+", " ", "\n".join(cur_text)).strip()
                if txt:
                    items.append((cur_id, txt))
            cur_id = p
            cur_text = []
        else:
            cur_text.append(p.strip())
    if cur_id is not None:
        txt = re.sub(r"\s+", " ", "\n".join(cur_text)).strip()
        if txt:
            items.append((cur_id, txt))
    cleaned = []
    for eid, txt in items:
        txt = re.sub(r"^Cluster profile.*?:\s*", "", txt, flags=re.IGNORECASE)
        txt = re.sub(r"^Patient snippet.*?:\s*", "", txt, flags=re.IGNORECASE)
        cleaned.append((eid, txt.strip()))
    return cleaned

def tfidf_cosine(a: str, b: str):
    v = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
    X = v.fit_transform([a, b])
    return float(cosine_similarity(X[0], X[1])[0,0])

def extract_claims(summary_text: str, min_chars=20, max_claims=15):
    lines = [ln.strip() for ln in summary_text.splitlines() if ln.strip()]
    bullet_lines = [ln for ln in lines if ln.startswith(("-", "•", "*"))]
    text_no_cite = re.sub(r"\[(?:C|S\d+)\]", "", summary_text)
    text_no_cite = re.sub(r"\s+", " ", text_no_cite).strip()
    claims = []
    if bullet_lines:
        for ln in bullet_lines:
            ln = re.sub(r"^[-•*]\s*", "", ln)
            ln = re.sub(r"\[(?:C|S\d+)\]", "", ln)
            ln = re.sub(r"\s+", " ", ln).strip()
            if len(ln) >= min_chars:
                claims.append(ln)
    else:
        sents = re.split(r"(?<=[\.\!\?])\s+", text_no_cite)
        claims = [s.strip() for s in sents if len(s.strip()) >= min_chars]
    if len(claims) > max_claims:
        claims = claims[:max_claims]
    return claims

def minmax_norm(a, eps=1e-9):
    a = np.asarray(a).astype(float)
    return (a - a.min())/(a.max()-a.min()+eps)

def support_check_claims(claims, evidence_items, sim_threshold=0.22):
    e_ids = [eid for eid,_ in evidence_items]
    e_texts = [txt for _,txt in evidence_items]
    v = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
    X = v.fit_transform(e_texts + claims)
    E = X[:len(e_texts)]
    C = X[len(e_texts):]
    sims = cosine_similarity(C, E)
    results = []
    for i, claim in enumerate(claims):
        j = int(np.argmax(sims[i]))
        best_sim = float(sims[i,j])
        results.append({
            "claim": claim,
            "supported": best_sim >= sim_threshold,
            "best_evidence_id": e_ids[j],
            "best_similarity": best_sim
        })
    return results

def pick_must_mentions(cluster_name):
    row = profiles.loc[profiles["risk_group"]==cluster_name].iloc[0]
    bp_best = max(BP_FAMILY, key=lambda c: row[c])
    met_best = max(MET_FAMILY, key=lambda c: row[c])
    return [bp_best, met_best]

def completeness_score(summary_text, must_inds):
    s = summary_text.lower()
    hits = 0
    for ind in must_inds:
        kws = INDICATOR_KEYWORDS.get(ind, [])
        if any(kw in s for kw in kws):
            hits += 1
    return hits / max(1, len(must_inds)), hits

def count_patterns(text, patterns):
    t = text.lower()
    total = 0
    for pat in patterns:
        total += len(re.findall(pat, t))
    return total

def evaluate_one(item, n_claims=12):
    cluster = item["cluster"]
    rag = item["rag_summary"]
    ref = item["reference_s1"]
    ev  = item["evidence_block"]
    sim = tfidf_cosine(rag, ref)

    evidence_items = parse_evidence_block(ev)
    claims = extract_claims(rag, max_claims=n_claims)
    cr = support_check_claims(claims, evidence_items)
    supported = sum(1 for r in cr if r["supported"])
    total = len(cr)
    supported_rate = (supported/total*100) if total else 0.0
    unsupported_rate = 100-supported_rate if total else 0.0

    must = pick_must_mentions(cluster)
    comp, comp_hits = completeness_score(rag, must)

    abs_ct = count_patterns(rag, ABSOLUTE_PATTERNS)
    diag_ct = count_patterns(rag, DIAGNOSIS_PATTERNS)
    med_ct = count_patterns(rag, MEDICATION_PATTERNS)

    return {
        "cluster": cluster,
        "similarity_tfidf": sim,
        "supported_claim_rate_%": supported_rate,
        "unsupported_claim_rate_%": unsupported_rate,
        "n_claims_checked": total,
        "completeness_%": comp*100,
        "must_mentions": ";".join(must),
        "must_mentions_hit": comp_hits,
        "absolute_claim_flags": abs_ct,
        "diagnosis_like_flags": diag_ct,
        "medication_flags": med_ct
    }

def main():
    with open("step7_eval_input_S1vsRAG.json","r",encoding="utf-8") as f:
        items = json.load(f)
    rows = [evaluate_one(it) for it in items]
    out = pd.DataFrame(rows)
    out.to_csv("step7_evaluation_results_S1vsRAG.csv", index=False)
    print(out)

if __name__ == "__main__":
    main()


## 11. Evaluate the RAG summaries with the S4 (t-SNE-PSO) reference

Run the same evaluator using `step7_eval_input_S4vsRAG.json`, where the reference is the original **S4 cluster-level summary** from Paper 1.

**Output:** `step7_evaluation_results_S4vsRAG.csv`.

Because both inputs use the **same RAG summary and evidence block**, their claim-support rates, completeness scores, and safety flags are expected to be identical. The S1 and S4 TF-IDF *reference-similarity* scores can differ.

In [ ]:
#!/usr/bin/env python3
import json, re
import pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROFILE_TABLE = "cluster_profile_table.csv"

profiles = pd.read_csv(PROFILE_TABLE)

BP_FAMILY = ["pct_BP_elevated_or_higher", "pct_BP_stage2"]
MET_FAMILY = ["pct_BMI_overweight", "pct_BMI_obese", "pct_Glucose_prediabetes", "pct_Glucose_diabetes", "pct_Chol_borderline", "pct_Chol_high"]

INDICATOR_KEYWORDS = {
    "pct_BP_elevated_or_higher": ["blood pressure", "bp", "hypertension", "pre-hypertens", "systolic", "diastolic", "elevated bp"],
    "pct_BP_stage2": ["stage 2", "severe hypertension", "very high blood pressure", "140/90"],
    "pct_BMI_overweight": ["overweight", "bmi", "weight"],
    "pct_BMI_obese": ["obesity", "obese", "bmi"],
    "pct_Glucose_prediabetes": ["glucose", "prediabet", "blood sugar", "elevated sugar"],
    "pct_Glucose_diabetes": ["diabetes", "diabetic", "hyperglyc", "very high glucose"],
    "pct_Chol_borderline": ["cholesterol", "borderline cholesterol", "lipid"],
    "pct_Chol_high": ["high cholesterol", "hypercholesterol", "cholesterol", "lipid"]
}

ABSOLUTE_PATTERNS = [r"\ball patients\b", r"\bevery patient\b", r"\ball individuals\b", r"\bnone of the patients\b", r"\balways\b", r"\bnever\b"]
DIAGNOSIS_PATTERNS = [r"\bdiagnos(ed|is)\b", r"\bhas (?:a|an|the)\b.*\b(disease|disorder|condition)\b", r"\bchronic kidney disease\b", r"\bckd\b", r"\bmyocardial infarction\b", r"\bstroke\b"]
MEDICATION_PATTERNS = [r"\bprescribe\b", r"\bmedication\b", r"\bdrug\b", r"\bdosage\b", r"\bmetformin\b", r"\bstatin\b", r"\binsulin\b", r"\bantihypertensive\b"]

def parse_evidence_block(evidence_block: str):
    pattern = r"(\[(?:C|S\d+)\])"
    parts = re.split(pattern, evidence_block)
    items = []
    cur_id = None
    cur_text = []
    for p in parts:
        if not p:
            continue
        if re.fullmatch(pattern, p):
            if cur_id is not None:
                txt = re.sub(r"\s+", " ", "\n".join(cur_text)).strip()
                if txt:
                    items.append((cur_id, txt))
            cur_id = p
            cur_text = []
        else:
            cur_text.append(p.strip())
    if cur_id is not None:
        txt = re.sub(r"\s+", " ", "\n".join(cur_text)).strip()
        if txt:
            items.append((cur_id, txt))
    cleaned = []
    for eid, txt in items:
        txt = re.sub(r"^Cluster profile.*?:\s*", "", txt, flags=re.IGNORECASE)
        txt = re.sub(r"^Patient snippet.*?:\s*", "", txt, flags=re.IGNORECASE)
        cleaned.append((eid, txt.strip()))
    return cleaned

def tfidf_cosine(a: str, b: str):
    v = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
    X = v.fit_transform([a, b])
    return float(cosine_similarity(X[0], X[1])[0,0])

def extract_claims(summary_text: str, min_chars=20, max_claims=15):
    lines = [ln.strip() for ln in summary_text.splitlines() if ln.strip()]
    bullet_lines = [ln for ln in lines if ln.startswith(("-", "•", "*"))]
    text_no_cite = re.sub(r"\[(?:C|S\d+)\]", "", summary_text)
    text_no_cite = re.sub(r"\s+", " ", text_no_cite).strip()
    claims = []
    if bullet_lines:
        for ln in bullet_lines:
            ln = re.sub(r"^[-•*]\s*", "", ln)
            ln = re.sub(r"\[(?:C|S\d+)\]", "", ln)
            ln = re.sub(r"\s+", " ", ln).strip()
            if len(ln) >= min_chars:
                claims.append(ln)
    else:
        sents = re.split(r"(?<=[\.\!\?])\s+", text_no_cite)
        claims = [s.strip() for s in sents if len(s.strip()) >= min_chars]
    if len(claims) > max_claims:
        claims = claims[:max_claims]
    return claims

def minmax_norm(a, eps=1e-9):
    a = np.asarray(a).astype(float)
    return (a - a.min())/(a.max()-a.min()+eps)

def support_check_claims(claims, evidence_items, sim_threshold=0.22):
    e_ids = [eid for eid,_ in evidence_items]
    e_texts = [txt for _,txt in evidence_items]
    v = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
    X = v.fit_transform(e_texts + claims)
    E = X[:len(e_texts)]
    C = X[len(e_texts):]
    sims = cosine_similarity(C, E)
    results = []
    for i, claim in enumerate(claims):
        j = int(np.argmax(sims[i]))
        best_sim = float(sims[i,j])
        results.append({
            "claim": claim,
            "supported": best_sim >= sim_threshold,
            "best_evidence_id": e_ids[j],
            "best_similarity": best_sim
        })
    return results

def pick_must_mentions(cluster_name):
    row = profiles.loc[profiles["risk_group"]==cluster_name].iloc[0]
    bp_best = max(BP_FAMILY, key=lambda c: row[c])
    met_best = max(MET_FAMILY, key=lambda c: row[c])
    return [bp_best, met_best]

def completeness_score(summary_text, must_inds):
    s = summary_text.lower()
    hits = 0
    for ind in must_inds:
        kws = INDICATOR_KEYWORDS.get(ind, [])
        if any(kw in s for kw in kws):
            hits += 1
    return hits / max(1, len(must_inds)), hits

def count_patterns(text, patterns):
    t = text.lower()
    total = 0
    for pat in patterns:
        total += len(re.findall(pat, t))
    return total

def evaluate_one(item, n_claims=12):
    cluster = item["cluster"]
    rag = item["rag_summary"]
    ref = item["reference_s1"]
    ev  = item["evidence_block"]
    sim = tfidf_cosine(rag, ref)

    evidence_items = parse_evidence_block(ev)
    claims = extract_claims(rag, max_claims=n_claims)
    cr = support_check_claims(claims, evidence_items)
    supported = sum(1 for r in cr if r["supported"])
    total = len(cr)
    supported_rate = (supported/total*100) if total else 0.0
    unsupported_rate = 100-supported_rate if total else 0.0

    must = pick_must_mentions(cluster)
    comp, comp_hits = completeness_score(rag, must)

    abs_ct = count_patterns(rag, ABSOLUTE_PATTERNS)
    diag_ct = count_patterns(rag, DIAGNOSIS_PATTERNS)
    med_ct = count_patterns(rag, MEDICATION_PATTERNS)

    return {
        "cluster": cluster,
        "similarity_tfidf": sim,
        "supported_claim_rate_%": supported_rate,
        "unsupported_claim_rate_%": unsupported_rate,
        "n_claims_checked": total,
        "completeness_%": comp*100,
        "must_mentions": ";".join(must),
        "must_mentions_hit": comp_hits,
        "absolute_claim_flags": abs_ct,
        "diagnosis_like_flags": diag_ct,
        "medication_flags": med_ct
    }

def main():
    with open("step7_eval_input_S4vsRAG.json","r",encoding="utf-8") as f:
        items = json.load(f)
    rows = [evaluate_one(it) for it in items]
    out = pd.DataFrame(rows)
    out.to_csv("step7_evaluation_results_S4vsRAG.csv", index=False)
    print(out)

if __name__ == "__main__":
    main()


## 12. Re-evaluate the non-RAG S1 and S4 baseline summaries

Produce **two baseline CSVs** using the same Step 7 evaluator, with original Paper 1 cluster summaries substituted into the `rag_summary` input field:

- `step7_evaluation_results_S1baseline.csv`: evaluate the **S1** (true-label) summary; compare its text similarity to **S1 itself**.
- `step7_evaluation_results_S4baseline.csv`: evaluate the **S4** (t-SNE-PSO) summary; compare its text similarity to **S1**.

The inherited field name `rag_summary` here means **"text being evaluated"**, not that the baseline uses RAG. Both baseline evaluations reuse the corresponding current evidence blocks. That facilitates metric comparisons, but may not faithfully represent distinct S1/S4 source-data partitions; verify the evidence/grouping correspondence before claiming full reproduction.

In [ ]:
#!/usr/bin/env python3
import json, re
import pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROFILE_TABLE = "cluster_profile_table.csv"

profiles = pd.read_csv(PROFILE_TABLE)

BP_FAMILY = ["pct_BP_elevated_or_higher", "pct_BP_stage2"]
MET_FAMILY = ["pct_BMI_overweight", "pct_BMI_obese", "pct_Glucose_prediabetes", "pct_Glucose_diabetes", "pct_Chol_borderline", "pct_Chol_high"]

INDICATOR_KEYWORDS = {
    "pct_BP_elevated_or_higher": ["blood pressure", "bp", "hypertension", "pre-hypertens", "systolic", "diastolic", "elevated bp"],
    "pct_BP_stage2": ["stage 2", "severe hypertension", "very high blood pressure", "140/90"],
    "pct_BMI_overweight": ["overweight", "bmi", "weight"],
    "pct_BMI_obese": ["obesity", "obese", "bmi"],
    "pct_Glucose_prediabetes": ["glucose", "prediabet", "blood sugar", "elevated sugar"],
    "pct_Glucose_diabetes": ["diabetes", "diabetic", "hyperglyc", "very high glucose"],
    "pct_Chol_borderline": ["cholesterol", "borderline cholesterol", "lipid"],
    "pct_Chol_high": ["high cholesterol", "hypercholesterol", "cholesterol", "lipid"]
}

ABSOLUTE_PATTERNS = [r"\ball patients\b", r"\bevery patient\b", r"\ball individuals\b", r"\bnone of the patients\b", r"\balways\b", r"\bnever\b"]
DIAGNOSIS_PATTERNS = [r"\bdiagnos(ed|is)\b", r"\bhas (?:a|an|the)\b.*\b(disease|disorder|condition)\b", r"\bchronic kidney disease\b", r"\bckd\b", r"\bmyocardial infarction\b", r"\bstroke\b"]
MEDICATION_PATTERNS = [r"\bprescribe\b", r"\bmedication\b", r"\bdrug\b", r"\bdosage\b", r"\bmetformin\b", r"\bstatin\b", r"\binsulin\b", r"\bantihypertensive\b"]

def parse_evidence_block(evidence_block: str):
    pattern = r"(\[(?:C|S\d+)\])"
    parts = re.split(pattern, evidence_block)
    items = []
    cur_id = None
    cur_text = []
    for p in parts:
        if not p:
            continue
        if re.fullmatch(pattern, p):
            if cur_id is not None:
                txt = re.sub(r"\s+", " ", "\n".join(cur_text)).strip()
                if txt:
                    items.append((cur_id, txt))
            cur_id = p
            cur_text = []
        else:
            cur_text.append(p.strip())
    if cur_id is not None:
        txt = re.sub(r"\s+", " ", "\n".join(cur_text)).strip()
        if txt:
            items.append((cur_id, txt))
    cleaned = []
    for eid, txt in items:
        txt = re.sub(r"^Cluster profile.*?:\s*", "", txt, flags=re.IGNORECASE)
        txt = re.sub(r"^Patient snippet.*?:\s*", "", txt, flags=re.IGNORECASE)
        cleaned.append((eid, txt.strip()))
    return cleaned

def tfidf_cosine(a: str, b: str):
    v = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
    X = v.fit_transform([a, b])
    return float(cosine_similarity(X[0], X[1])[0,0])

def extract_claims(summary_text: str, min_chars=20, max_claims=15):
    lines = [ln.strip() for ln in summary_text.splitlines() if ln.strip()]
    bullet_lines = [ln for ln in lines if ln.startswith(("-", "•", "*"))]
    text_no_cite = re.sub(r"\[(?:C|S\d+)\]", "", summary_text)
    text_no_cite = re.sub(r"\s+", " ", text_no_cite).strip()
    claims = []
    if bullet_lines:
        for ln in bullet_lines:
            ln = re.sub(r"^[-•*]\s*", "", ln)
            ln = re.sub(r"\[(?:C|S\d+)\]", "", ln)
            ln = re.sub(r"\s+", " ", ln).strip()
            if len(ln) >= min_chars:
                claims.append(ln)
    else:
        sents = re.split(r"(?<=[\.\!\?])\s+", text_no_cite)
        claims = [s.strip() for s in sents if len(s.strip()) >= min_chars]
    if len(claims) > max_claims:
        claims = claims[:max_claims]
    return claims

def minmax_norm(a, eps=1e-9):
    a = np.asarray(a).astype(float)
    return (a - a.min())/(a.max()-a.min()+eps)

def support_check_claims(claims, evidence_items, sim_threshold=0.22):
    e_ids = [eid for eid,_ in evidence_items]
    e_texts = [txt for _,txt in evidence_items]
    v = TfidfVectorizer(stop_words="english", ngram_range=(1,2))
    X = v.fit_transform(e_texts + claims)
    E = X[:len(e_texts)]
    C = X[len(e_texts):]
    sims = cosine_similarity(C, E)
    results = []
    for i, claim in enumerate(claims):
        j = int(np.argmax(sims[i]))
        best_sim = float(sims[i,j])
        results.append({
            "claim": claim,
            "supported": best_sim >= sim_threshold,
            "best_evidence_id": e_ids[j],
            "best_similarity": best_sim
        })
    return results

def pick_must_mentions(cluster_name):
    row = profiles.loc[profiles["risk_group"]==cluster_name].iloc[0]
    bp_best = max(BP_FAMILY, key=lambda c: row[c])
    met_best = max(MET_FAMILY, key=lambda c: row[c])
    return [bp_best, met_best]

def completeness_score(summary_text, must_inds):
    s = summary_text.lower()
    hits = 0
    for ind in must_inds:
        kws = INDICATOR_KEYWORDS.get(ind, [])
        if any(kw in s for kw in kws):
            hits += 1
    return hits / max(1, len(must_inds)), hits

def count_patterns(text, patterns):
    t = text.lower()
    total = 0
    for pat in patterns:
        total += len(re.findall(pat, t))
    return total

def evaluate_one(item, n_claims=12):
    cluster = item["cluster"]
    rag = item["rag_summary"]
    ref = item["reference_s1"]
    ev  = item["evidence_block"]
    sim = tfidf_cosine(rag, ref)

    evidence_items = parse_evidence_block(ev)
    claims = extract_claims(rag, max_claims=n_claims)
    cr = support_check_claims(claims, evidence_items)
    supported = sum(1 for r in cr if r["supported"])
    total = len(cr)
    supported_rate = (supported/total*100) if total else 0.0
    unsupported_rate = 100-supported_rate if total else 0.0

    must = pick_must_mentions(cluster)
    comp, comp_hits = completeness_score(rag, must)

    abs_ct = count_patterns(rag, ABSOLUTE_PATTERNS)
    diag_ct = count_patterns(rag, DIAGNOSIS_PATTERNS)
    med_ct = count_patterns(rag, MEDICATION_PATTERNS)

    return {
        "cluster": cluster,
        "similarity_tfidf": sim,
        "supported_claim_rate_%": supported_rate,
        "unsupported_claim_rate_%": unsupported_rate,
        "n_claims_checked": total,
        "completeness_%": comp*100,
        "must_mentions": ";".join(must),
        "must_mentions_hit": comp_hits,
        "absolute_claim_flags": abs_ct,
        "diagnosis_like_flags": diag_ct,
        "medication_flags": med_ct
    }


from copy import deepcopy

def main():

    # Load the two JSON inputs created in the previous cell
    with open("step7_eval_input_S1vsRAG.json",
              "r", encoding="utf-8") as f:
        s1_items = json.load(f)

    with open("step7_eval_input_S4vsRAG.json",
              "r", encoding="utf-8") as f:
        s4_items = json.load(f)

    # Organize by risk group
    s1_by_cluster = {x["cluster"]: x for x in s1_items}
    s4_by_cluster = {x["cluster"]: x for x in s4_items}

    expected = {"low_risk", "average_risk", "high_risk"}

    if set(s1_by_cluster) != expected or set(s4_by_cluster) != expected:
        raise ValueError("Missing or unexpected risk groups.")

    s1_baseline_items = []
    s4_baseline_items = []

    for cluster in ["low_risk", "average_risk", "high_risk"]:

        s1 = s1_by_cluster[cluster]
        s4 = s4_by_cluster[cluster]

        # Original overall summaries from Paper 1
        s1_summary = s1["reference_s1"]
        s4_summary = s4["reference_s1"]

        # ----------------------------------------
        # S1 baseline: evaluate the original true-label cluster summary.
        # ----------------------------------------
        item_s1 = deepcopy(s1)

        # The evaluator reads the text to evaluate
        # from the field named "rag_summary".
        # Here, that field contains S1, NOT RAG.
        item_s1["rag_summary"] = s1_summary
        item_s1["reference_s1"] = s1_summary

        s1_baseline_items.append(item_s1)

        # ----------------------------------------
        # S4 baseline: evaluate the original t-SNE-PSO cluster summary.
        # ----------------------------------------
        item_s4 = deepcopy(s4)

        # Evaluate the original S4 summary
        item_s4["rag_summary"] = s4_summary

        # Compare S4 text similarity against S1
        item_s4["reference_s1"] = s1_summary

        s4_baseline_items.append(item_s4)

    # Run the same evaluation for both baselines
    s1_results = pd.DataFrame([
        evaluate_one(item) for item in s1_baseline_items
    ])

    s4_results = pd.DataFrame([
        evaluate_one(item) for item in s4_baseline_items
    ])

    # Save both baseline result tables in the Colab working directory.
    s1_results.to_csv(
        "step7_evaluation_results_S1baseline.csv",
        index=False
    )

    s4_results.to_csv(
        "step7_evaluation_results_S4baseline.csv",
        index=False
    )

    print("\n=== S1 BASELINE ===")
    print(s1_results.to_string(index=False))

    print("\n=== S4 BASELINE ===")
    print(s4_results.to_string(index=False))

    print("\nBoth baseline CSV files saved successfully.")

if __name__ == "__main__":
    main()



## 13. Calculate baseline-to-RAG differences (deltas)

The final cell reads the four Step 7 CSVs and computes per-cluster **differences between the current RAG scores and the respective original non-RAG summary scores**.

- `ΔSupported(%) = RAG − baseline` (**positive** is better).
- `ΔUnsupported(%) = baseline − RAG` (**positive** represents a reduction in unsupported claims).
- `ΔCompleteness(%) = RAG − baseline` (**positive** is better).
- `ΔAbsoluteFlags = baseline − RAG` (**positive** means fewer detected categorical phrases).

The percentage differences are **percentage points**, not relative percent changes. These are **automated evaluator outputs**, not independent clinical correctness judgments.

In [ ]:
import pandas as pd

rag_s1 = pd.read_csv("step7_evaluation_results_S1vsRAG.csv").set_index("cluster")
base_s1 = pd.read_csv("step7_evaluation_results_S1baseline.csv").set_index("cluster")

rag_s4 = pd.read_csv("step7_evaluation_results_S4vsRAG.csv").set_index("cluster")
base_s4 = pd.read_csv("step7_evaluation_results_S4baseline.csv").set_index("cluster")

def delta(rag, base, label):
    out = pd.DataFrame(index=rag.index)
    out["Comparison"] = label
    out["ΔSupported(%)"] = rag["supported_claim_rate_%"] - base["supported_claim_rate_%"]
    out["ΔUnsupported(%)"] = base["unsupported_claim_rate_%"] - rag["unsupported_claim_rate_%"]
    out["ΔCompleteness(%)"] = rag["completeness_%"] - base["completeness_%"]
    out["ΔAbsoluteFlags"] = base["absolute_claim_flags"] - rag["absolute_claim_flags"]
    return out.reset_index()

print(pd.concat([
    delta(rag_s1, base_s1, "S1 baseline → S1-RAG"),
    delta(rag_s4, base_s4, "S4 baseline → S4-RAG"),
], ignore_index=True))


---

## Interpretation and reproducibility notes

**Scope of evidence:** the repository accompanies research on synthetic hypertension records. The source data, retrieval method, generation prompt, and evaluator should be inspected together before attributing a score change to the retrieval weight alone.

**Faithfulness metric:** the `supported_claim_rate_%` column is based on claim-to-evidence **TF-IDF overlap** using a fixed threshold; it may misclassify a true claim that paraphrases a numeric fact. `completeness_%` and safety flags rely on keyword/pattern matching.

**Published vs. new results:** Paper 2 reports α = 0.25, but the best alpha is recomputed in each run. Report the selected alpha, exact prompts, evidence items, model/embedding identifiers, generation parameters, and evaluation setup when sharing new results. Do not overwrite the published tables with new-run scores without labeling them.

**Current implementation caveats:** the cluster profile may be retrieved twice; the S1- and S4-reference evaluations share the same generated RAG summaries and evidence; and the three final generation calls can differ from the sweep outputs even when `temperature=0`.

**Files to retain for an auditable run:** `alpha_sweep_results.csv`, `alpha_sweep_macro.csv`, `alpha_sweep_outputs.json`, both `step7_eval_input_*.json` files, four `step7_evaluation_results_*.csv` files, and the full input file versions. API calls may incur costs and are not guaranteed deterministic.

**Citation:** K. Torabizadeh, R. Hedjam, M. Allaoui, and B. Abdulrazak, *Cluster-Aware Retrieval-Augmented Generation (RAG) with Hybrid Retrieval for Faithful Medical Report Summarization*, Proceedings of Machine Learning Research 318, 2026. [Official publication](https://proceedings.mlr.press/v318/torabizadeh26a.html).